# agentic-sql on Colab

Runtime > Change runtime type > T4 GPU.

The repo is the source of truth. This notebook only clones it, installs, and
calls the CLI. Results go to Drive so a disconnect does not lose them, and
rerunning any cell resumes from the per-question checkpoint.

In [ ]:
REPO = "https://github.com/USERNAME/agentic-sql.git"  # <- your repo

import os
from google.colab import drive
drive.mount('/content/drive')

RESULTS = '/content/drive/MyDrive/agentic-sql/results'
os.makedirs(RESULTS, exist_ok=True)

if not os.path.exists('/content/agentic-sql'):
    !git clone -q $REPO /content/agentic-sql
%cd /content/agentic-sql
!git pull -q
!pip -q install -r requirements.txt

os.environ['PYTHONPATH'] = '/content/agentic-sql/src'
os.environ['AGSQL_RESULTS'] = RESULTS
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Build and check the dataset

Downloads the Chinook SQL, builds the database, asserts every gold query still
executes and returns the row count the dataset records.

In [ ]:
!python data/verify_dataset.py

## Smoke test first

One question per arm. Do not skip this. It takes two minutes and catches the
failures that otherwise surface an hour into the grid.

In [ ]:
!python -m agsql.cli arms
!python -m agsql.cli run --arm all --smoke

## The grid

Cheap arms first, so a run that dies partway still leaves most of the chart.
Keep this tab in the foreground.

In [ ]:
!python -m agsql.cli run --arm all

## Tables

In [ ]:
!python -m agsql.cli report --failures